# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

This interactive workbook demonstrates example of Elasticsearch's [MultiQuery Retriever](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) to generate similar queries for a given user input and apply all queries to retrieve a larger set of relevant documents from a vectorstore.

Before we begin, we first split the fictional workplace documents into passages with `langchain` and uses OpenAI to transform these passages into embeddings and then store these into Elasticsearch.

We will then ask a question, generate similar questions using langchain and OpenAI, retrieve relevant passages from the vector store, and use langchain and OpenAI again to provide a summary for the questions.

## Install packages and import modules

In [1]:
!python3 -m pip install -qU jq lark langchain langchain-elasticsearch langchain_openai tiktoken

from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai.llms import OpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass

'python3' is not recognized as an internal or external command,
operable program or batch file.


## Connect to Elasticsearch

ℹ️ We're using an Elastic Cloud deployment of Elasticsearch for this notebook. If you don't have an Elastic Cloud deployment, sign up [here](https://cloud.elastic.co/registration?utm_source=github&utm_content=elasticsearch-labs-notebook) for a free trial. 

We'll use the **Cloud ID** to identify our deployment, because we are using Elastic Cloud deployment. To find the Cloud ID for your deployment, go to https://cloud.elastic.co/deployments and select your deployment.

We will use [ElasticsearchStore](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html) to connect to our elastic cloud deployment, This would help create and index data easily.  We would also send list of documents that we created in the previous step

In [16]:
# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#finding-your-cloud-id
ELASTIC_CLOUD_ID = getpass("Elastic Cloud ID: ")

# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#creating-an-api-key
ELASTIC_API_KEY = getpass("Elastic Api Key: ")

# https://platform.openai.com/api-keys
OPENAI_API_KEY = getpass("OpenAI API key: ")

embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name="langchain-old",
    embedding=embeddings,
)

## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [17]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [ ]:
from langchain_community.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
import elasticsearch

es = elasticsearch.Elasticsearch(cloud_id= ELASTIC_CLOUD_ID, api_key= ELASTIC_API_KEY)
es.indices.delete(index='langchain-old', ignore_unavailable=True)


def metadata_func(record: dict, metadata: dict) -> dict:
    # Populate metadata with relevant fields from the JSON record
    metadata["name"] = record.get("name", " ") 
    metadata["summary"] = record.get("summary", " ")
    metadata["url"] = record.get("url", " ")
    metadata["category"] = record.get("category", " ")
    metadata["updated_at"] = record.get("updated_at", " ")
    return metadata

# Loader
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

# Text splitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500, chunk_overlap=50
)

# Load and split the documents
docs = loader.load_and_split(text_splitter=text_splitter)


### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [39]:
docs[0].metadata

{'source': 'C:\\Users\\nicho\\Documents\\ironhack\\Week 18\\lab-chatbot-with-multi-query-retriever\\temp.json',
 'seq_num': 1,
 'name': 'Work From Home Policy',
 'summary': 'This policy outlines the guidelines for full-time remote work, including eligibility, equipment and resources, workspace requirements, communication expectations, performance expectations, time tracking and overtime, confidentiality and data security, health and well-being, and policy reviews and updates. Employees are encouraged to direct any questions or concerns',
 'url': './sharepoint/Work from home policy.txt',
 'category': 'teams',
 'updated_at': '2020-03-01'}

In [40]:
docs[0]

Document(metadata={'source': 'C:\\Users\\nicho\\Documents\\ironhack\\Week 18\\lab-chatbot-with-multi-query-retriever\\temp.json', 'seq_num': 1, 'name': 'Work From Home Policy', 'summary': 'This policy outlines the guidelines for full-time remote work, including eligibility, equipment and resources, workspace requirements, communication expectations, performance expectations, time tracking and overtime, confidentiality and data security, health and well-being, and policy reviews and updates. Employees are encouraged to direct any questions or concerns', 'url': './sharepoint/Work from home policy.txt', 'category': 'teams', 'updated_at': '2020-03-01'}, page_content="Effective: March 2020\nPurpose\n\nThe purpose of this full-time work-from-home policy is to provide guidelines and support for employees to conduct their work remotely, ensuring the continuity and productivity of business operations during the COVID-19 pandemic and beyond.\nScope\n\nThis policy applies to all employees who are

In [50]:

for i, doc in enumerate(docs):
    try:
        vectorstore = ElasticsearchStore.from_documents(
            [doc],
            embeddings,
            index_name="langchain-old",
            es_cloud_id=ELASTIC_CLOUD_ID,
            es_api_key=ELASTIC_API_KEY,
            )
    except Exception as e:
        print('error', e)
        print('i:', i)


error 1 document(s) failed to index.
i: 8
error 1 document(s) failed to index.
i: 9
error 1 document(s) failed to index.
i: 10
error 1 document(s) failed to index.
i: 11
error 1 document(s) failed to index.
i: 12
error 1 document(s) failed to index.
i: 13
error 1 document(s) failed to index.
i: 14
error 1 document(s) failed to index.
i: 15
error 1 document(s) failed to index.
i: 16
error 1 document(s) failed to index.
i: 17
error 1 document(s) failed to index.
i: 18
error 1 document(s) failed to index.
i: 19
error 1 document(s) failed to index.
i: 20
error 1 document(s) failed to index.
i: 21
error 1 document(s) failed to index.
i: 22
error 1 document(s) failed to index.
i: 23
error 1 document(s) failed to index.
i: 24
error 1 document(s) failed to index.
i: 25
error 1 document(s) failed to index.
i: 26


In [51]:
llm = OpenAI(temperature=0, openai_api_key=OPENAI_API_KEY)

retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [53]:
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document

import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("what is the strategy for the fiscal year of 2024?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. What are the plans for the fiscal year of 2024?', '2. How is the fiscal year of 2024 being approached?', '3. What is the proposed strategy for the fiscal year of 2024?']


---- Answer ----
The strategy for the fiscal year of 2024 is outlined in the sales strategy document for the tech company. The primary goal is to increase revenue, expand market share, and strengthen customer relationships in target markets. This will be achieved through various objectives, focus areas, and action plans, including increasing revenue by 20%, expanding market share by 15%, retaining 95% of existing customers, and launching at least two new products or services in high-demand market segments. The focus areas include targeting high-growth industries, strengthening relationships with key accounts and strategic partners, and optimizing the product/service portfolio. The action plans involve expanding the sales team, implementing targeted marketing campaigns, strengthening partnerships, and implementing a proactive customer success program. The strategy will also be regularly monitored and evaluated through key performance indicators and quarterly reviews to ensure alignment 

**Generate at least two new iteratioins of the previous cells - Be creative.** Did you master Multi-
Query Retriever concepts through this lab?

In [54]:
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document

import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("What was the annual revenue of 2024?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. What was the total income in 2024?', '2. How much money did the company make in 2024?', '3. What was the yearly earnings in 2024?']


---- Answer ----
The annual revenue for fiscal year 2024 is projected to increase by 20% compared to fiscal year 2023, according to the company's sales strategy document. This is one of the key objectives outlined in the document, along with expanding market share and strengthening customer relationships. The company also plans to launch at least two new products or services in high-demand market segments in order to achieve this revenue growth. The sales team will be expanded and provided with ongoing training, and targeted marketing campaigns will be developed to reach different customer segments and industries. Regular monitoring and evaluation will also be conducted to track progress and make adjustments as needed.


In [56]:
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document

import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("what is the work from home policy")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. What are the guidelines for working remotely?', '2. Can you explain the policy for telecommuting?', '3. How does the company handle remote work arrangements?']


---- Answer ----

The work from home policy is a set of guidelines and support provided by a company to allow employees to work remotely. It was implemented in March 2020 in response to the COVID-19 pandemic and is designed to ensure the continuity and productivity of business operations. The policy applies to eligible employees who have received approval from their supervisor and the HR department. It includes provisions for necessary equipment and resources, workspace requirements, communication expectations, work hours and availability, performance expectations, time tracking and overtime, confidentiality and data security, and employee health and well-being. The policy is periodically reviewed and updated as needed, and employees are encouraged to direct any questions or concerns to their supervisor or the HR department.
